# ETL

### Importar librerias

In [2]:
import pandas as pd
import matplotlib as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR

### Cargar datos

In [4]:
df = pd.read_csv("data.csv")
df.head()

,video_id,author,description,likes,comments,shares,plays,hashtags,music,create_time,video_url,fetch_time,views,posted_time
0,7506183500660313390,dalebrisby90,"That sounds like JB, ima go with TRUTH! 🤔🤣 #ro...",44800,125,1863,686000.0,"rodeotime, dalebrisby, jbmauney",original sound,2025-05-19 15:45:53,https://www.tiktok.com/@dalebrisby90/video/750...,NaN,NaN,NaN
1,7507316543605280030,jessicafloriolli,thanks for sticking around @Alonzofloriolli,285700,290,963,1200000.0,NaN,ECE Marketing Airball,2025-05-22 17:02:36,https://www.tiktok.com/@jessicafloriolli/video...,NaN,NaN,NaN
2,7507286333505719582,ay_2fya,#publicinterview #fyp #rizz,38000,65,496,416100.0,"publicinterview, fyp, rizz",original sound,2025-05-22 15:05:21,https://www.tiktok.com/@ay_2fya/video/75072863...,NaN,NaN,NaN
3,7506662216574209310,abell1823,Boat days hit different ✨,87200,259,23600,725800.0,NaN,If You Were Mine,2025-05-20 22:43:34,https://www.tiktok.com/@abell1823/video/750666...,NaN,NaN,NaN
4,7506628206280363310,jordanmarielynnxoxo,a lululemonnnn. #prettygirl #foryoupage #trend...,77600,724,1196,362100.0,"prettygirl, foryoupage, trending, likes",What Da Fuk,2025-05-20 20:32:12,https://www.tiktok.com/@jordanmarielynnxoxo/vi...,NaN,NaN,NaN


### Contar nulos por columna

In [5]:
df.isnull().sum()

video_id          0
author            0
description     692
likes             0
comments          0
shares            0
plays             7
hashtags       2075
music             0
create_time       7
video_url         0
fetch_time     7218
views          7218
posted_time    7218
dtype: int64

### Eliminar fetch_time, views, posted_time. No hay manera de llenarlos porque todos san vacios

In [7]:
df = df.drop("fetch_time", axis=1)
df = df.drop("views", axis=1)
df = df.drop("posted_time", axis=1)

KeyError: "['fetch_time'] not found in axis"

In [8]:
df.head()

,video_id,author,description,likes,comments,shares,plays,hashtags,music,create_time,video_url
0,7506183500660313390,dalebrisby90,"That sounds like JB, ima go with TRUTH! 🤔🤣 #ro...",44800,125,1863,686000.0,"rodeotime, dalebrisby, jbmauney",original sound,2025-05-19 15:45:53,https://www.tiktok.com/@dalebrisby90/video/750...
1,7507316543605280030,jessicafloriolli,thanks for sticking around @Alonzofloriolli,285700,290,963,1200000.0,NaN,ECE Marketing Airball,2025-05-22 17:02:36,https://www.tiktok.com/@jessicafloriolli/video...
2,7507286333505719582,ay_2fya,#publicinterview #fyp #rizz,38000,65,496,416100.0,"publicinterview, fyp, rizz",original sound,2025-05-22 15:05:21,https://www.tiktok.com/@ay_2fya/video/75072863...
3,7506662216574209310,abell1823,Boat days hit different ✨,87200,259,23600,725800.0,NaN,If You Were Mine,2025-05-20 22:43:34,https://www.tiktok.com/@abell1823/video/750666...
4,7506628206280363310,jordanmarielynnxoxo,a lululemonnnn. #prettygirl #foryoupage #trend...,77600,724,1196,362100.0,"prettygirl, foryoupage, trending, likes",What Da Fuk,2025-05-20 20:32:12,https://www.tiktok.com/@jordanmarielynnxoxo/vi...


### Inputar valores nulos

#### inputar play con modelos de ML

In [10]:
temp = df.dropna(subset=["plays", "likes", "comments", "shares"])

X = temp[["likes", "comments", "shares"]]
y = temp["plays"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


models = {
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Linear Regression": LinearRegression(),
    "KNN": KNeighborsRegressor(n_neighbors=5),
    "Neural Network (MLP)": MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42),
    "SVM (SVR)": SVR()
}


results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    score = r2_score(y_test, preds)
    results[name] = score

# =========================
# 4. RESULTADOS
# =========================
for name, score in results.items():
    print(f"{name}: R² = {score:.4f}")


Random Forest: R² = 0.6012
Linear Regression: R² = 0.6402
KNN: R² = 0.5503
Neural Network (MLP): R² = 0.6326
SVM (SVR): R² = -0.0843


In [13]:
df_train = df[df["plays"].notnull()]
df_missing = df[df["plays"].isnull()]

X_train = df_train[["likes", "comments", "shares"]]
y_train = df_train["plays"]

X_missing = df_missing[["likes", "comments", "shares"]]

model = LinearRegression()
model.fit(X_train, y_train)

df.loc[df["plays"].isnull(), "plays"] = model.predict(X_missing)
df.isnull().sum()

ValueError: Found array with 0 sample(s) (shape=(0, 3)) while a minimum of 1 is required by LinearRegression.

#### Inputar create_time

In [14]:
df["create_time"] = pd.to_datetime(df["create_time"])
mean_value = df["create_time"].mean()
df["create_time"] = df["create_time"].fillna(mean_value)
df.isnull().sum()

video_id          0
author            0
description     692
likes             0
comments          0
shares            0
plays             0
hashtags       2075
music             0
create_time       0
video_url         0
dtype: int64

### Castear tipos de datos

#### Tipo string

In [15]:
# author, video_url, description, hashtags, music
df["author"] = df["author"].astype("string")
df["video_url"] = df["video_url"].astype("string")
df["description"] = df["description"].astype("string")
df["hashtags"] = df["hashtags"].astype("string")
df["music"] = df["music"].astype("string")

#### Tipo numerico

In [16]:
# video_id, likes, comments, shares, plays
df["video_id"] = df["video_id"].astype("int64")
df["likes"] = df["likes"].astype("int64")
df["comments"] = df["comments"].astype("int64")
df["shares"] = df["shares"].astype("int64")
df["plays"] = df["plays"].astype("int64")

#### Tipo date

In [17]:
# create_time
df["create_time"] = pd.to_datetime(df["create_time"])
